In [1]:
!pip install pyspark -q
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, datetime, gc
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Cấu hình đường dẫn
BASE_PATH = "/content/drive/MyDrive/HM-DATA/"
INPUT_FILE = BASE_PATH + "processed/cleaned_transactions.parquet"
OUTPUT_DIR = BASE_PATH + "outputs/candidates/"
MODEL_PATH = BASE_PATH + "outputs/models/xgb_ranker_final.json"

# 2. Khởi tạo Spark tối ưu CPU
spark = SparkSession.builder \
    .appName("HM_Final_Inference_W8_v6Features") \
    .config("spark.driver.memory", "12g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

# 3. Xác định mốc Tuần 8 (7 ngày cuối cùng)
transactions = spark.read.parquet(INPUT_FILE)
max_date = transactions.select(F.max("t_dat")).collect()[0][0]
test_start_date = max_date - datetime.timedelta(days=7)

print(f"✅ Spark Ready! Đang thực hiện dự đoán Tuần 8 dựa trên mô hình 6 đặc trưng.")

Mounted at /content/drive
✅ Spark Ready! Đang thực hiện dự đoán Tuần 8 dựa trên mô hình 6 đặc trưng.


In [2]:
print("⏳ Đang chuẩn bị Mapping Article ID...")
articles = spark.read.parquet(BASE_PATH + "processed/articles_processed.parquet")
mapping_7to10 = articles.select("article_id") \
    .withColumn("product_code", F.substring(F.col("article_id"), 1, 7)) \
    .groupBy("product_code").agg(F.first("article_id").alias("article_id_10")).cache()

print(f"✅ Đã chuẩn bị xong mapping cho {mapping_7to10.count():,} sản phẩm.")

⏳ Đang chuẩn bị Mapping Article ID...
✅ Đã chuẩn bị xong mapping cho 47,224 sản phẩm.


In [3]:
def load_sources_w8():
    # A. Các nguồn cần Explode (Dạng folder)
    hist = spark.read.parquet(OUTPUT_DIR + "history_candidates_W8.parquet") \
                .select("customer_id", F.explode("history_candidates").alias("article_id"))

    meta = spark.read.parquet(OUTPUT_DIR + "meta_candidates_pro_W8.parquet") \
                .select("customer_id", F.explode("meta_candidates").alias("article_id"))

    trend = spark.read.parquet(OUTPUT_DIR + "trending_candidates_W8.parquet") \
                 .select("customer_id", F.explode("trending_candidates").alias("article_id"))

    fp = spark.read.parquet(OUTPUT_DIR + "fp_association_candidates_val_W8.parquet") \
              .select("customer_id", F.explode("fp_candidates").alias("product_code")) \
              .join(F.broadcast(mapping_7to10), "product_code") \
              .select("customer_id", F.col("article_id_10").alias("article_id"))

    # B. Các nguồn đọc trực tiếp (Dạng file - Tham khảo TrainCpu ô 3)
    # File ALS là folder nhưng chứa dữ liệu phẳng
    als = spark.read.parquet(OUTPUT_DIR + "als_top100_W8_decoded.parquet")

    # File IMAGE là file đơn lẻ (Tham khảo TrainCpu.ipynb)
    image = spark.read.parquet(OUTPUT_DIR + "image_candidates_W8.parquet").repartition(200)

    return hist, meta, trend, fp, als, image

h8, m8, t8, f8, a8, i8 = load_sources_w8()

# C. Union tổng lực 6 nguồn (Tham khảo TrainCpu ô 4)
all_union = h8.select("customer_id", "article_id") \
    .union(m8.select("customer_id", "article_id")) \
    .union(t8.select("customer_id", "article_id")) \
    .union(f8.select("customer_id", "article_id")) \
    .union(a8.select("customer_id", "article_id")) \
    .union(i8.select("customer_id", "article_id"))

candidates_w8 = all_union.distinct()

# D. Join điểm số cho tầng Ranking
candidates_w8 = candidates_w8.join(a8, ["customer_id", "article_id"], "left").fillna(0, subset=["als_score"])
candidates_w8 = candidates_w8.join(i8, ["customer_id", "article_id"], "left").fillna(0, subset=["clip_score"])

print(f"📊 Tổng cộng: {candidates_w8.count():,} ứng viên Tuần 8 đã hội quân.")

📊 Tổng cộng: 72,834,509 ứng viên Tuần 8 đã hội quân.


In [4]:
print("🛠️ Đang tính toán 6 đặc trưng cho Tuần 8...")

# 1. Đặc trưng Sản phẩm (Popularity & Price tính đến hết Tuần 7)
item_feat = transactions.filter(F.col("t_dat") < F.lit(test_start_date)) \
    .groupBy("article_id").agg(F.count("customer_id").alias("item_popularity"), F.avg("price").alias("item_price")) \
    .withColumn("article_id", F.lpad(F.col("article_id").cast("string"), 10, "0"))

# 2. Đặc trưng Khách hàng (Budget & Frequency tính đến hết Tuần 7)
user_feat = transactions.filter(F.col("t_dat") < F.lit(test_start_date)) \
    .groupBy(F.col("customer_id").cast("string")).agg(F.avg("price").alias("user_avg_budget"), F.count("article_id").alias("user_buy_freq"))

# 3. Hợp nhất bảng Inference
full_inference_set = candidates_w8.join(F.broadcast(item_feat), "article_id", "left") \
                                   .join(F.broadcast(user_feat), "customer_id", "left").fillna(0)

print("✅ Đã chuẩn bị xong dữ liệu với 6 đặc trưng chuẩn.")

🛠️ Đang tính toán 6 đặc trưng cho Tuần 8...
✅ Đã chuẩn bị xong dữ liệu với 6 đặc trưng chuẩn.


In [5]:
import xgboost as xgb

# Danh sách 6 đặc trưng khớp hoàn toàn với mô hình hiện tại của Leader
features = ["als_score", "clip_score", "item_popularity", "item_price", "user_avg_budget", "user_buy_freq"]

def predict_batch_w8(iterator):
    import xgboost as xgb
    bst = xgb.Booster()
    bst.load_model(MODEL_PATH) # Nạp file xgb_ranker_final.json
    bst.set_param({'predictor': 'cpu_predictor'})
    for pdf in iterator:
        pdf['pred_score'] = bst.predict(xgb.DMatrix(pdf[features]))
        yield pdf[['customer_id', 'article_id', 'pred_score']]

print("🚀 Đang tiến hành chấm điểm cho Tuần 8 (CPU Mode)...")
predictions = full_inference_set.repartition(800).mapInPandas(predict_batch_w8,
    schema="customer_id string, article_id string, pred_score float")

# Xếp hạng và lấy Top 12
top_12_w8 = predictions.withColumn("rn", F.row_number().over(Window.partitionBy("customer_id").orderBy(F.desc("pred_score")))) \
    .filter(F.col("rn") <= 12).groupBy("customer_id").agg(F.collect_list("article_id").alias("prediction"))

top_12_w8.cache().count()
print("✅ Đã hoàn tất dự đoán cho Tuần 8.")

🚀 Đang tiến hành chấm điểm cho Tuần 8 (CPU Mode)...
✅ Đã hoàn tất dự đoán cho Tuần 8.


In [7]:
from pyspark.mllib.evaluation import RankingMetrics

print("🎯 Đang tổng hợp bộ chỉ số báo cáo cho Tuần 8...")

# 1. Chuẩn bị dữ liệu đánh giá (Chỉ tính trên khách hàng có mua hàng ở Tuần 8)
eval_df = top_12_w8.join(actual_w8, "customer_id", "inner").select("prediction", "actual")
eval_rdd = eval_df.rdd.map(lambda r: (list(r[0]), list(r[1])))

# Khởi tạo công cụ đo lường
metrics = RankingMetrics(eval_rdd)

# 2. Tính toán "Bộ 3 chỉ số vàng"
# MAP@12: Chỉ số xếp hạng chuẩn (Càng cao thì món đúng nằm càng ở đầu danh sách)
map_12 = metrics.meanAveragePrecision

# Recall@12: Chỉ số bao phủ (Trong 10 món khách mua, AI 'vớt' được bao nhiêu món)
recall_12 = metrics.recallAt(12)

# Hit Rate: Chỉ số kinh doanh (Tỷ lệ khách hàng hài lòng - có ít nhất 1 món trúng)
hit_count = eval_df.rdd.map(lambda r: 1 if len(set(r[0]) & set(r[1])) > 0 else 0).sum()
total_users = eval_df.count()
hit_rate = (hit_count / total_users) * 100 if total_users > 0 else 0

# 3. XUẤT BẢNG TÓM TẮT BÁO CÁO
print("\n" + "="*50)
print(f"📊 KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH (TUẦN CUỐI - TEST SET)")
print("-" * 50)
print(f"1. MAP@12 (Độ chính xác xếp hạng):  {map_12:.6f}")
print(f"2. Recall@12 (Khả năng bắt trúng):  {recall_12:.6f}")
print(f"3. Hit Rate (Tỷ lệ khách hài lòng): {hit_rate:.2f}%")
print("-" * 50)
print(f"👉 Ý nghĩa: Cứ 100 khách mua hàng, AI đoán trúng ")
print(f"   ít nhất 1 món cho {hit_rate:.1f} người.")
print("="*50)

🎯 Đang tổng hợp bộ chỉ số báo cáo cho Tuần 8...

📊 KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH (TUẦN CUỐI - TEST SET)
--------------------------------------------------
1. MAP@12 (Độ chính xác xếp hạng):  0.009628
2. Recall@12 (Khả năng bắt trúng):  0.028272
3. Hit Rate (Tỷ lệ khách hài lòng): 6.44%
--------------------------------------------------
👉 Ý nghĩa: Cứ 100 khách mua hàng, AI đoán trúng 
   ít nhất 1 món cho 6.4 người.
